# Importers MinIO Test

导入 Windows 图片目录 `C:\\Users\\wuchaoli\\Pictures\\测试图片`，并写入 MinIO 受管图片库。Notebook 运行产物统一写入 `notebooks/.importers_test_library/minio_outputs/`。MinIO 连接参数从仓库根目录 `.env` 读取。


In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

from image_gallery.importers import ImportPipeline, LocalDirectoryReader
from image_gallery.storage import MinioStorage


repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
load_dotenv(repo_root / ".env")


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


raw_endpoint = require_env("IMAGE_GALLERY_MINIO_ENDPOINT")
access_key = require_env("IMAGE_GALLERY_MINIO_ACCESS_KEY")
secret_key = require_env("IMAGE_GALLERY_MINIO_SECRET_KEY")
bucket = require_env("IMAGE_GALLERY_MINIO_BUCKET")

parsed = urlparse(raw_endpoint)
endpoint = parsed.netloc or raw_endpoint
secure = parsed.scheme == "https"

windows_source_dir = Path(r"C:\Users\wuchaoli\Pictures\测试图片")
wsl_source_dir = Path("/mnt/c/Users/wuchaoli/Pictures/测试图片")
source_dir = wsl_source_dir if wsl_source_dir.exists() else windows_source_dir
if not source_dir.exists():
    raise FileNotFoundError(f"source directory not found: {source_dir}")

library_root = repo_root / "notebooks" / ".importers_test_library"
output_dir = library_root / "minio_outputs"
records = LocalDirectoryReader(source_dir).read()

print("source_dir:", source_dir)
print("output_dir:", output_dir)
print("bucket:", bucket)
print("secure:", secure)
print("source records:", len(records))


In [ ]:
storage = MinioStorage(storage_name="minio_test_picture_library").connect(
    endpoint=endpoint,
    access_key=access_key,
    secret_key=secret_key,
    bucket=bucket,
    secure=secure,
)

print("connected bucket:", storage.bucket)
print("storage_name:", storage.storage_name)


In [ ]:
result = ImportPipeline(
    storage=storage,
    output_dir=output_dir,
    global_tags=["dataset/test_pictures", "source/windows_pictures", "storage/minio"],
).run(records)

print("raw_dataset_path:", result.raw_dataset_path)
print("import_report_path:", result.import_report_path)
print("failure_manifest_path:", result.failure_manifest_path)
print("report:", result.report)


In [ ]:
import uuid
from datetime import datetime
from pathlib import PurePosixPath

from image_gallery.dataset import Dataset


raw_frame = Dataset.from_path(result.raw_dataset_path).to_frame()
today = datetime.now().date().isoformat()
image_uri_prefix = f"s3://{bucket}/images/raw/"

if raw_frame.empty:
    raise AssertionError("raw dataset should contain imported rows")
if not raw_frame["image_uri"].str.startswith(image_uri_prefix).all():
    raise AssertionError(f"image_uri should start with {image_uri_prefix}")
if not raw_frame["image_uri"].str.contains(f"images/raw/{today}/shard_").all():
    raise AssertionError("image_uri should contain today's date shard path")
if not (raw_frame["storage_name"] == "minio_test_picture_library").all():
    raise AssertionError("storage_name should be minio_test_picture_library")

for row in raw_frame.itertuples(index=False):
    object_name = str(row.image_uri).removeprefix(f"s3://{bucket}/")
    managed_name = PurePosixPath(object_name).name
    uuid.UUID(PurePosixPath(managed_name).stem)
    if managed_name == row.source_file_name:
        raise AssertionError("managed MinIO object should not expose source_file_name")
    if not managed_name.endswith(PurePosixPath(row.source_file_name).suffix.lower()):
        raise AssertionError("managed MinIO object should preserve lowercase source extension")

print("raw rows:", len(raw_frame))
print(raw_frame.head())


In [ ]:
sample_rows = raw_frame.head(5)
readback = []
for row in sample_rows.itertuples(index=False):
    object_path = str(row.image_uri).removeprefix(f"s3://{bucket}/")
    data = storage.read_bytes(object_path)
    if not data:
        raise AssertionError(f"empty object data: {object_path}")
    readback.append({"object_path": object_path, "bytes": len(data)})

readback


In [ ]:
preview_columns = [
    "source_file_name",
    "image_uri",
    "image_format",
    "width",
    "height",
    "file_size_bytes",
]
raw_frame[preview_columns].head(20)
